In [6]:
# pairwise comparison of msas, so each collection of msa is unique.

from Bio import SeqIO
import os
import shutil  # Import shutil for file operations

def compare_sequence_files(file1_path, file2_path, output_dir):
    """
    Compare sequences between two files and output unique and common sequences.
    """
    file1_base = os.path.splitext(os.path.basename(file1_path))[0]
    file2_base = os.path.splitext(os.path.basename(file2_path))[0]

    # Parse sequences from both files
    file1_seqs = {record.id: record for record in SeqIO.parse(file1_path, "fasta")}
    file2_seqs = {record.id: record for record in SeqIO.parse(file2_path, "fasta")}

    # Determine common and unique sequences
    common_ids = set(file1_seqs.keys()) & set(file2_seqs.keys())
    unique_file1_ids = set(file1_seqs.keys()) - common_ids
    unique_file2_ids = set(file2_seqs.keys()) - common_ids

    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # File paths for output
    output_files = {
        'common': os.path.join(output_dir, f"{file2_base}{file1_base}.fasta"),  # Concatenated name
        'unique1': os.path.join(output_dir, f"{file1_base}.fasta"),  # Updated file1
        'unique2': os.path.join(output_dir, f"{file2_base}0.fasta"),  # Updated file2
    }

    # Write outputs
    for key, seq_ids in [('common', common_ids), ('unique1', unique_file1_ids), ('unique2', unique_file2_ids)]:
        output_path = output_files[key]
        with open(output_path, 'w') as output_handle:
            if key == 'unique1':
                SeqIO.write((file1_seqs[id_] for id_ in seq_ids), output_handle, "fasta")
            elif key == 'unique2':
                SeqIO.write((file2_seqs[id_] for id_ in seq_ids), output_handle, "fasta")
            else:
                SeqIO.write((file1_seqs[id_] for id_ in common_ids), output_handle, "fasta")

        # Remove empty files
        if os.path.getsize(output_path) == 0:
            os.remove(output_path)
            print(f"Deleted empty file: {output_path}")

    # Overwrite file1 with unique sequences for next comparison
    with open(file1_path, 'w') as output_handle:
        SeqIO.write((file1_seqs[id_] for id_ in unique_file1_ids), output_handle, "fasta")

    # Write a summary
    summary_file = os.path.join(output_dir, f"{file1_base}_{file2_base}_summary.txt")
    with open(summary_file, 'w') as f:
        f.write(f"Total sequences in {file1_base}: {len(file1_seqs)}\n")
        f.write(f"Total sequences in {file2_base}: {len(file2_seqs)}\n")
        f.write(f"Common sequences: {len(common_ids)}\n")
        f.write(f"Unique sequences in {file1_base}: {len(unique_file1_ids)}\n")
        f.write(f"Unique sequences in {file2_base}: {len(unique_file2_ids)}\n")

    print(f"Comparison completed for {file1_base} and {file2_base}. Outputs saved to {output_dir}.")

def process_files_in_rounds(file_list, base_folder):
    """
    Process files sequentially, comparing each file with all files in the previous round's folder,
    creating a new folder for each round.
    """
    round_number = 1

    # Use the first file as the initial reference
    initial_file = file_list.pop(0)
    current_round_folder = os.path.join(base_folder, f"Combine_Round{round_number}")
    os.makedirs(current_round_folder, exist_ok=True)
    shutil.copy(initial_file, os.path.join(current_round_folder, os.path.basename(initial_file)))

    while file_list:
        next_file = file_list.pop(0)
        next_file_name = os.path.basename(next_file)
        next_file_path = os.path.join(base_folder, next_file_name)

        # Create the folder for the next round
        next_round_folder = os.path.join(base_folder, f"Combine_Round{round_number + 1}")
        os.makedirs(next_round_folder, exist_ok=True)

        # Compare the new file with all files in the current round's folder
        for fasta_file in os.listdir(current_round_folder):
            if fasta_file.endswith('.fasta'):  # Only process FASTA files
                fasta_path = os.path.join(current_round_folder, fasta_file)
                compare_sequence_files(next_file_path, fasta_path, next_round_folder)

        # Update the new file path after it is overwritten with unique sequences
        next_file_path = os.path.join(next_round_folder, f"{os.path.splitext(next_file_name)[0]}.fasta")

        # Move to the next round
        current_round_folder = next_round_folder
        round_number += 1

# Example Usage
combine_folder = "/home/yuhong/demo/combine_All_MSA"  # Path to the Combine folder
file_list = [
    "/home/yuhong/demo/combine_All_MSA/1.fasta",
    "/home/yuhong/demo/combine_All_MSA/2.fasta",
    "/home/yuhong/demo/combine_All_MSA/3.fasta",
    "/home/yuhong/demo/combine_All_MSA/4.fasta",
    "/home/yuhong/demo/combine_All_MSA/7.fasta",
    "/home/yuhong/demo/combine_All_MSA/8.fasta",
]

process_files_in_rounds(file_list, combine_folder)


Comparison completed for 2 and 1. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round2.
Comparison completed for 3 and 10. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round3.
Deleted empty file: /home/yuhong/demo/combine_All_MSA/Combine_Round3/123.fasta
Comparison completed for 3 and 12. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round3.
Deleted empty file: /home/yuhong/demo/combine_All_MSA/Combine_Round3/23.fasta
Comparison completed for 3 and 2. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round3.
Deleted empty file: /home/yuhong/demo/combine_All_MSA/Combine_Round4/1034.fasta
Comparison completed for 4 and 103. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round4.
Deleted empty file: /home/yuhong/demo/combine_All_MSA/Combine_Round4/34.fasta
Comparison completed for 4 and 3. Outputs saved to /home/yuhong/demo/combine_All_MSA/Combine_Round4.
Deleted empty file: /home/yuhong/demo/combine_All_MSA/Combine_Round4/1004.f